In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Importing the dataset

In [ ]:
from pathlib import Path

DATA_CANDIDATES = (Path("data/Concrete_Data_Yeh.csv"), Path("Concrete_Data_Yeh.csv"))
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not included. Place Concrete_Data_Yeh.csv in the repository root "
        "or in the data/ directory."
    )
df = pd.read_csv(DATA_PATH)

EXPECTED_COLUMNS = {
    "cement", "slag", "flyash", "water", "superplasticizer",
    "coarseaggregate", "fineaggregate", "age", "csMPa"
}
missing_columns = EXPECTED_COLUMNS.difference(df.columns)
if missing_columns:
    raise ValueError("Dataset is missing required columns: {}".format(sorted(missing_columns)))

## Data Description

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

##  Build a Dashboard for Stakeholders

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='csMPa', bins=30, kde=True, color='r')
plt.title('Distribution of Concrete Strength (csMPa)', fontsize=14)
plt.xlabel('Concrete Strength (MPa)', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='age', y='csMPa', color='green')
plt.title('Concrete Strength by Age', fontsize=14)
plt.xlabel('Age (days)', fontsize=14)
plt.ylabel('Concrete Strength (MPa)', fontsize=14)
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='water', y='csMPa', color='blue')
plt.title('Water vs Concrete Strength', fontsize=14)
plt.xlabel('Water (kg)', fontsize=14)
plt.ylabel('Concrete Strength (MPa)', fontsize=14)
plt.grid(True)
plt.show()

## Applying  Models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
X = df.drop("csMPa", axis=1)
y = df["csMPa"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(" Model evaluation:")
print(f"R² Score: {r2:.2f}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})
print(comparison.head(10))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(y_test.values[:20], label='Actual', marker='o')
plt.plot(y_pred[:20], label='Predicted', marker='x')
plt.title('Actual vs Predicted Concrete Strength')
plt.xlabel('Sample Index')
plt.ylabel('Compressive Strength (MPa)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None
    print("XGBoost is not installed; skipping the optional XGBoost model.")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
rf_pred = rf_model.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print(" Random Forest Results:")
print(f"R² Score: {rf_r2:.2f}")
print(f"MAE: {rf_mae:.2f}")
print(f"RMSE: {rf_rmse:.2f}")

In [ ]:
if XGBRegressor is not None:
    xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    xgb_model.fit(X_train, y_train)

In [ ]:
if XGBRegressor is not None:
    xgb_pred = xgb_model.predict(X_test)
    xgb_r2 = r2_score(y_test, xgb_pred)
    xgb_mae = mean_absolute_error(y_test, xgb_pred)
    xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

    print("XGBoost results:")
    print(f"R² Score: {xgb_r2:.2f}")
    print(f"MAE: {xgb_mae:.2f}")
    print(f"RMSE: {xgb_rmse:.2f}")